# Script 3.1 — Painel de Validação e Predições por Empresa

**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Roda imediatamente após o `03_cvm_modelagem.ipynb`. Consome os modelos
treinados e os artefatos de predição para gerar o painel completo de
resultados por empresa — a principal referência da banca para avaliar
o desempenho do pipeline.

| Bloco | Conteúdo |
|-------|----------|
| 1 | Tabela de validação hold-out 2024–2025: real vs. predito vs. erro por empresa |
| 2 | Evolução temporal completa: histórico + predição 2026 por empresa e setor |
| 3 | KPIs derivados das predições (Opção A — Penman 2013) com nota metodológica |
| 4 | Z\'\'-Score prospectivo calculado sobre os targets preditos |
| 5 | Painel comparativo intra-setor: todas as empresas lado a lado |

**Saídas:**
- `outputs/painel/` — todos os arquivos deste script
- `b31_validacao_holdout.csv`            — erro por empresa × target (hold-out)
- `b31_validacao_holdout_{setor}.png`    — real vs. predito visual
- `b31_predicao_2026_empresa.csv`        — predições 2026 por empresa × target
- `b31_kpis_derivados_2026.csv`          — KPIs calculados das predições
- `b31_zscore_prospectivo.csv`           — Z\'\' sobre predições 2026
- `b31_painel_setor_{setor}.png`         — comparativo intra-setor

## Etapa 0. Imports e Configuração

In [10]:
import json, logging, pickle, warnings
from pathlib import Path
from datetime import datetime

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)

PASTA_SAIDA = Path('outputs')
PASTA_PAINEL = PASTA_SAIDA / 'painel'
PASTA_PAINEL.mkdir(exist_ok=True)

logger = logging.getLogger('script_31')
logger.setLevel(logging.INFO)
logger.handlers.clear()
_sh = logging.StreamHandler()
_sh.setFormatter(logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s', datefmt='%H:%M:%S'))
logger.addHandler(_sh)
_fh = logging.FileHandler(PASTA_SAIDA / 'logs' / 'script_31.log', mode='w', encoding='utf-8')
_fh.setFormatter(logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s'))
logger.addHandler(_fh)

# ── Constantes espelhadas do Script 3 ────────────────────────────────────────
_TARGET_BASES = ['DRE_3.01','DRE_3.11','EBITDA',
                 'BPA_1','BPA_1.01','BPP_2.01','BPP_2.03','BPP_2','DFC_MI_6.01']
_HORIZONTES   = ['_ITR_T1','_ITR_T2','_ITR_T3','_DFP']
_LOG_BASES     = {'DRE_3.01','EBITDA','BPA_1','BPA_1.01','BPP_2.01','BPP_2.03','BPP_2'}
_ARCSINH_BASES = {'DFC_MI_6.01','DRE_3.11'}
LOG_TARGETS    = {f'TARGET_{b}{h}' for b in _LOG_BASES     for h in _HORIZONTES}
ARCSINH_TARGETS= {f'TARGET_{b}{h}' for b in _ARCSINH_BASES for h in _HORIZONTES}

NOME_VAR = {
    'DRE_3.01':'Receita Líquida','DRE_3.11':'Lucro Líquido','EBITDA':'EBITDA',
    'BPA_1':'Ativo Total','BPA_1.01':'Ativo Circulante','BPP_2.01':'Passivo Circulante',
    'BPP_2.03':'Patrimônio Líquido','BPP_2':'Passivo Total','DFC_MI_6.01':'FCO',
}
CORES_SETOR = {
    'Petróleo':'#1f4e79','Energia':'#2e75b6',
    'Varejo':'#ed7d31','Commodities':'#70ad47','Tecnologia':'#ffc000',
}

def inv_transform(y, target):
    y = np.asarray(y, float)
    if target in LOG_TARGETS:     return np.expm1(y)
    if target in ARCSINH_TARGETS: return np.sinh(y)
    return y

def smape(yt, yp):
    yt, yp = np.asarray(yt,float), np.asarray(yp,float)
    d = (np.abs(yt)+np.abs(yp))/2.0
    m = d > 1e-9
    return float(np.mean(np.abs(yt[m]-yp[m])/d[m])) if m.sum()>0 else np.nan

logger.info("Script 3.1 iniciado em %s", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("✅ Etapa 0 concluída")

22:18:51 | INFO     | Script 3.1 iniciado em 2026-06-01 22:18:51


✅ Etapa 0 concluída


## Etapa 1. Carga dos Artefatos do Script 3

In [11]:
with open(PASTA_SAIDA / 'melhores_modelos.pkl', 'rb') as f:
    melhores = pickle.load(f)
with open(PASTA_SAIDA / 'selected_features_por_target.pkl', 'rb') as f:
    selected_features = pickle.load(f)

treino  = pd.read_parquet(PASTA_SAIDA / 'treino.parquet')
teste   = pd.read_parquet(PASTA_SAIDA / 'teste.parquet')
dataset = pd.read_parquet(PASTA_SAIDA / 'dataset_cvm_consolidado.parquet')

_p = PASTA_SAIDA / 'predicoes_teste_detalhadas.parquet'
df_pred_det = pd.read_parquet(_p) if _p.exists() else pd.DataFrame()
_p2 = PASTA_SAIDA / 'predicoes_prospectivas.parquet'
df_prosp    = pd.read_parquet(_p2) if _p2.exists() else pd.DataFrame()

TARGETS = [f'TARGET_{b}{h}' for b in _TARGET_BASES for h in _HORIZONTES
           if f'TARGET_{b}{h}' in melhores]

# Recria coluna SETOR se necessário
if 'SETOR' not in dataset.columns:
    setor_cols = [c for c in dataset.columns if c.startswith('setor_')]
    if setor_cols:
        dataset['SETOR'] = (dataset[setor_cols].idxmax(axis=1)
                            .str.replace('setor_',''))

# Mapas de lookup
mapa_nome  = {}; mapa_setor = {}; mapa_cnpj = {}
if 'CNPJ_CIA' in dataset.columns:
    _b = dataset.drop_duplicates('CNPJ_CIA')
    if 'NOME_CIA' in _b.columns:
        mapa_nome  = _b.set_index('CNPJ_CIA')['NOME_CIA'].to_dict()
        mapa_cnpj  = _b.set_index('NOME_CIA')['CNPJ_CIA'].to_dict()
    if 'SETOR' in _b.columns:
        mapa_setor = _b.set_index('CNPJ_CIA')['SETOR'].to_dict()

setores_empresas = {}
for cnpj, setor in mapa_setor.items():
    setores_empresas.setdefault(setor, []).append(cnpj)

print(f"Targets: {len(TARGETS)} | Empresas: {len(mapa_nome)} | Setores: {len(setores_empresas)}")
print(f"Predições hold-out: {len(df_pred_det):,} | Predições prospectivas: {len(df_prosp):,}")
logger.info("Artefatos carregados | targets=%d | empresas=%d", len(TARGETS), len(mapa_nome))

22:18:52 | INFO     | Artefatos carregados | targets=36 | empresas=25


Targets: 36 | Empresas: 25 | Setores: 5
Predições hold-out: 28,035 | Predições prospectivas: 828


## Bloco 1. Tabela de Validação Hold-out 2024–2025

**Real vs. Predito por empresa, target e horizonte.**

Esta é a tabela principal para a banca: mostra o valor real observado
no hold-out, o valor predito pelo melhor modelo e os erros absoluto e
percentual simétrico (SMAPE). Valores reais existem → comparação direta.

*Referência: Hyndman & Athanasopoulos (2018) — avaliação em horizonte fixo.*

In [12]:
print("\n" + "="*70)
print("  BLOCO 1 - Tabela de Validacao Hold-out 2024-2025")
print("="*70)

rows_val = []

if not df_pred_det.empty:
    df_pd = df_pred_det.copy()
    df_pd['NOME_CIA'] = df_pd['CNPJ_CIA'].map(mapa_nome)
    df_pd['SETOR']    = df_pd['CNPJ_CIA'].map(mapa_setor)

    df_best = df_pd[
        df_pd.apply(lambda r: melhores.get(r['Target']) == r['Algoritmo'], axis=1)
    ].copy()

    # Detecta se y_true/y_pred estao na escala original ou transformada.
    # Se a mediana absoluta for > 1e4, os valores ja estao em R$ (escala original)
    # e NAO devemos aplicar inv_transform novamente.
    # O Script 3 tipicamente salva as predicoes JA invertidas para escala original.
    _sample_abs = df_best['y_true'].dropna().abs()
    _median_abs = float(_sample_abs.median()) if len(_sample_abs) > 0 else 0.0
    _already_original = _median_abs > 1e4

    logger.info("Hold-out: %d linhas | mediana |y_true|=%.2e | escala=%s",
                len(df_best), _median_abs,
                'ORIGINAL (sem inv_transform)' if _already_original else 'TRANSFORMADA (aplica inv_transform)')

    for _, row in df_best.iterrows():
        target = row['Target']
        yt_raw = row['y_true']
        yp_raw = row['y_pred']

        if pd.isna(yt_raw) or pd.isna(yp_raw):
            continue

        if _already_original:
            yt = float(yt_raw)
            yp = float(yp_raw)
        else:
            yt = float(inv_transform([yt_raw], target)[0])
            yp = float(inv_transform([yp_raw], target)[0])

        if np.isnan(yt) or np.isnan(yp) or np.isinf(yt) or np.isinf(yp):
            logger.warning("Valor invalido pos-transform | target=%s | empresa=%s | yt=%s yp=%s",
                           target, row.get('NOME_CIA',''), yt_raw, yp_raw)
            continue

        erro_abs = abs(yt - yp)
        denom    = (abs(yt) + abs(yp)) / 2.0
        smape_v  = float(erro_abs / denom) if denom > 1e-9 else np.nan

        base = target.replace('TARGET_','').rsplit('_ITR',1)[0].rsplit('_DFP',1)[0]
        hor  = next((h for h in _HORIZONTES if target.endswith(h)), 'N/A')

        rows_val.append({
            'empresa':     row.get('NOME_CIA', ''),
            'setor':       row.get('SETOR', ''),
            'target':      target,
            'variavel':    NOME_VAR.get(base, base),
            'horizonte':   hor.replace('_', ''),
            'algoritmo':   row['Algoritmo'],
            'y_real_bi':   round(yt / 1e6, 4),
            'y_pred_bi':   round(yp / 1e6, 4),
            'erro_abs_bi': round(erro_abs / 1e6, 4),
            'smape':       round(smape_v, 4) if not np.isnan(smape_v) else None,
            'dt_refer':    row.get('DT_REFER', ''),
        })

    df_val = pd.DataFrame(rows_val).sort_values(['setor','empresa','variavel','horizonte'])
    df_val.to_csv(PASTA_PAINEL / 'b31_validacao_holdout.csv', index=False)
    print(f"  OK b31_validacao_holdout.csv ({len(df_val):,} linhas)")

    print("\n  SMAPE medio por variavel x horizonte (hold-out, escala original):")
    df_val_smape = df_val.dropna(subset=['smape'])
    n_nan = df_val['smape'].isna().sum()
    if not df_val_smape.empty:
        resumo = (df_val_smape
                  .groupby(['variavel','horizonte'])['smape']
                  .agg(['mean','median','count'])
                  .round(4))
        print(resumo.to_string())
        if n_nan:
            print(f"\n  INFO: {n_nan} linhas com SMAPE=NaN excluidas (real~0 e pred~0).")
    else:
        print("  ATENCAO: Nenhuma linha com SMAPE valido.")
        print(f"  _already_original={_already_original} | mediana |y_true|={_median_abs:.4g}")
        print("  Verifique a escala em que y_true/y_pred foram salvos no Script 3.")
else:
    df_val = pd.DataFrame()
    print("  AVISO: predicoes_teste_detalhadas.parquet nao encontrado.")

print("\n  OK Bloco 1 concluido")



  BLOCO 1 - Tabela de Validacao Hold-out 2024-2025


22:18:52 | INFO     | Hold-out: 5607 linhas | mediana |y_true|=1.13e+07 | escala=ORIGINAL (sem inv_transform)


  OK b31_validacao_holdout.csv (5,607 linhas)

  SMAPE medio por variavel x horizonte (hold-out, escala original):
                               mean  median  count
variavel           horizonte                      
Ativo Circulante   DFP       0.1760  0.0990    165
                   ITRT1     0.1214  0.0907    193
                   ITRT2     0.1123  0.0803    145
                   ITRT3     0.0873  0.0670    120
Ativo Total        DFP       0.1830  0.0764    165
                   ITRT1     0.0997  0.0671    193
                   ITRT2     0.0751  0.0479    145
                   ITRT3     0.0645  0.0466    120
EBITDA             DFP       0.1113  0.0709    165
                   ITRT1     0.2713  0.1196    193
                   ITRT2     0.1596  0.1163    145
                   ITRT3     0.0983  0.0658    120
FCO                DFP       0.7743  0.4826    165
                   ITRT1     1.2265  1.3737    193
                   ITRT2     1.0765  0.9618    145
                  

### Bloco 1b. Figuras — Real vs. Predito por Setor

In [13]:
# Grafico por setor x variavel foco - scatter real vs. predito
BASES_FOCO = ['DRE_3.01','DRE_3.11','EBITDA']

if not df_val.empty:
    for setor in sorted(df_val['setor'].dropna().unique()):
        df_s = df_val[(df_val['setor'] == setor) &
                      (df_val['variavel'].isin([NOME_VAR[b] for b in BASES_FOCO]))]
        if df_s.empty:
            continue

        variaveis  = [NOME_VAR[b] for b in BASES_FOCO if NOME_VAR[b] in df_s['variavel'].unique()]
        n_var      = len(variaveis)
        fig, axes  = plt.subplots(1, n_var, figsize=(5*n_var, 5), squeeze=False)
        empresas_s = sorted(df_s['empresa'].dropna().unique())
        cores_emp  = plt.cm.tab10(np.linspace(0, 1, max(len(empresas_s), 1)))
        emp_cor    = {e: cores_emp[i] for i, e in enumerate(empresas_s)}

        for i_v, var in enumerate(variaveis):
            ax   = axes[0][i_v]
            df_v = df_s[df_s['variavel'] == var].dropna(subset=['y_pred_bi','y_real_bi'])

            for emp in empresas_s:
                sub = df_v[df_v['empresa'] == emp]
                if sub.empty: continue
                ax.scatter(sub['y_pred_bi'], sub['y_real_bi'],
                           color=emp_cor[emp], alpha=0.75, s=40,
                           label=emp[:12], zorder=3)

            x_vals = df_v['y_pred_bi']
            y_vals = df_v['y_real_bi']
            if not x_vals.empty and not y_vals.empty:
                lim_min = min(x_vals.min(), y_vals.min())
                lim_max = max(x_vals.max(), y_vals.max())
                ax.plot([lim_min, lim_max], [lim_min, lim_max],
                        'k--', lw=1.2, alpha=0.6, label='Identidade (y=x)')

            smape_vals = df_v['smape'].dropna()
            if not smape_vals.empty:
                titulo_smape = f"SMAPE medio = {smape_vals.mean():.1%}"
            else:
                titulo_smape = "SMAPE = N/D"
            ax.set_title(f'{var}\n{titulo_smape}', fontsize=9, fontweight='bold')
            ax.set_xlabel('Predito (R$ bi)', fontsize=8)
            ax.set_ylabel('Real (R$ bi)', fontsize=8)
            ax.legend(fontsize=6, loc='upper left', ncol=2)
            ax.grid(alpha=0.25)

        plt.suptitle(f'Validacao Hold-out - {setor} (2024-2025)',
                     fontsize=11, fontweight='bold')
        plt.tight_layout()
        fname = f'b31_validacao_holdout_{setor.replace(" ","_")}.png'
        plt.savefig(PASTA_PAINEL / fname, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  OK {fname}")

print("  OK Bloco 1b concluido")


  OK b31_validacao_holdout_Commodities.png
  OK b31_validacao_holdout_Energia.png
  OK b31_validacao_holdout_Petróleo.png
  OK b31_validacao_holdout_Tecnologia.png
  OK b31_validacao_holdout_Varejo.png
  OK Bloco 1b concluido


## Bloco 2. Predições 2026 por Empresa

Aplica os melhores modelos sobre o último vetor de KPIs disponível
de cada empresa para gerar o valor predito no horizonte DFP 2026.

**Distinção metodológica (conforme seções 4.5.1 e 4.5.2 do TCC):**
A predição parte de dados *reais* observados (KPIs do ITR Q1/2026 ou
DFP 2025). Não é uma projeção hipotética — é a estimativa do modelo
para o próximo período com base no estado financeiro atual da empresa.

In [14]:
print("\n" + "="*70)
print("  BLOCO 2 - Predicoes DFP 2026 por Empresa")
print("="*70)

TARGETS_DFP = [t for t in TARGETS if t.endswith('_DFP')]

# CORRECAO PRINCIPAL:
# O codigo anterior tentava re-predizer montando o vetor x com apenas as
# features filtradas por dataset.columns (~10-14 features), mas o pipeline
# do modelo foi treinado com 150-158 features (one-hot, lags, etc.).
# Resultado: erro "X has N features, but SimpleImputer expecting M" para todas as empresas.
#
# SOLUCAO: usar df_prosp (predicoes_prospectivas.parquet) que o Script 3 ja
# gerou com o pipeline correto. Filtramos pelo horizonte DFP e o ano mais recente.

rows_pred = []

if not df_prosp.empty:
    df_p = df_prosp.copy()
    df_p.columns = df_p.columns.str.strip()

    logger.info("df_prosp: %d linhas | colunas: %s", len(df_p), list(df_p.columns))

    # Identifica colunas de forma tolerante a variações de nome do Script 3
    col_map = {}
    for c in df_p.columns:
        cl = c.lower().strip()
        if cl in ('target', 'variavel_target', 'target_name', 'nome_target'):
            col_map['target'] = c
        elif cl in ('y_pred', 'pred', 'predicao', 'y_pred_original', 'valor_pred', 'y_predito'):
            col_map['pred'] = c
        elif cl in ('cnpj_cia', 'cnpj', 'cd_cvm'):
            col_map['cnpj'] = c
        elif cl in ('ano', 'ano_pred', 'year', 'exercicio'):
            col_map['ano'] = c
        elif cl in ('algoritmo', 'algorithm', 'model', 'modelo'):
            col_map['alg'] = c

    logger.info("Mapeamento de colunas df_prosp: %s", col_map)

    _required = ['target', 'pred', 'cnpj']
    _missing  = [k for k in _required if k not in col_map]

    if _missing:
        print(f"  ATENCAO: colunas nao encontradas em df_prosp: {_missing}")
        print(f"  Colunas disponiveis: {list(df_p.columns)}")
        print("  Primeiras linhas:")
        print(df_p.head(3).to_string())
        df_pred26 = pd.DataFrame()
    else:
        c_tgt = col_map['target']
        c_prd = col_map['pred']
        c_cnpj = col_map['cnpj']
        c_ano  = col_map.get('ano')
        c_alg  = col_map.get('alg')

        # Filtra apenas targets DFP
        mask_dfp = df_p[c_tgt].astype(str).str.endswith('_DFP', na=False)
        df_dfp = df_p[mask_dfp].copy()

        if df_dfp.empty:
            print(f"  ATENCAO: nenhum target _DFP em df_prosp.")
            print(f"  Targets presentes: {df_p[c_tgt].unique()[:10]}")
            df_pred26 = pd.DataFrame()
        else:
            # Pega o registro mais recente por empresa+target (preferencialmente 2026)
            if c_ano is not None:
                df_dfp[c_ano] = pd.to_numeric(df_dfp[c_ano], errors='coerce')
                df_dfp = (df_dfp.sort_values(c_ano)
                                .groupby([c_cnpj, c_tgt], as_index=False)
                                .last())

            for _, row in df_dfp.iterrows():
                cnpj     = row[c_cnpj]
                target   = str(row[c_tgt])
                y_pred_v = row[c_prd]
                alg_nome = str(row[c_alg]) if c_alg else melhores.get(target, '')

                if pd.isna(y_pred_v):
                    continue

                y_pred_f = float(y_pred_v)
                if np.isnan(y_pred_f) or np.isinf(y_pred_f):
                    continue

                # Detecta se o valor esta em escala transformada pela magnitude:
                # valores em log1p de bilhoes ficam entre ~14 e ~25;
                # valores em escala original (R$) ficam na casa dos bilhoes (> 1e8)
                if abs(y_pred_f) < 100:
                    if target in LOG_TARGETS:
                        y_pred_f = float(np.expm1(y_pred_f))
                    elif target in ARCSINH_TARGETS:
                        y_pred_f = float(np.sinh(y_pred_f))

                if np.isnan(y_pred_f) or np.isinf(y_pred_f):
                    continue

                nome_emp = mapa_nome.get(cnpj, str(cnpj))
                setor    = mapa_setor.get(cnpj, '')
                base     = target.replace('TARGET_','').replace('_DFP','')
                ano_base = 2026
                if c_ano and not pd.isna(row.get(c_ano)):
                    ano_base = int(row[c_ano])

                rows_pred.append({
                    'cnpj':           cnpj,
                    'empresa':        nome_emp,
                    'setor':          setor,
                    'ano_base':       ano_base,
                    'target':         target,
                    'variavel':       NOME_VAR.get(base, base),
                    'algoritmo':      alg_nome,
                    'y_pred_2026_bi': round(y_pred_f / 1e6, 4),
                })

            df_pred26 = (pd.DataFrame(rows_pred).sort_values(['setor','empresa','variavel'])
                         if rows_pred else pd.DataFrame())

else:
    # df_prosp vazio: fallback com padding de features para o pipeline
    logger.warning("df_prosp vazio - tentando predicao direta com padding de features")

    def ultimos_kpis_empresa(cnpj):
        df_e = dataset[dataset['CNPJ_CIA'] == cnpj]
        if df_e.empty: return None, None
        df_e = df_e.sort_values('ANO') if 'ANO' in df_e.columns else df_e
        ul  = df_e.iloc[-1]
        ano = int(ul['ANO']) if 'ANO' in ul.index else None
        return ul, ano

    for cnpj in mapa_nome:
        nome_emp     = mapa_nome[cnpj]
        setor        = mapa_setor.get(cnpj, '')
        ul, ano_base = ultimos_kpis_empresa(cnpj)
        if ul is None: continue

        for target in TARGETS_DFP:
            if target not in melhores: continue
            alg_nome = melhores[target]
            cam = PASTA_SAIDA / 'modelos' / f'modelo_{target}_{alg_nome}.pkl'
            if not cam.exists(): continue

            try:
                obj    = joblib.load(cam)
                modelo = obj['modelo'] if isinstance(obj, dict) else obj
            except Exception:
                continue

            # Descobre quantas features o pipeline espera
            _n_exp = None
            try:
                for _step in (getattr(modelo, 'steps', None) or []):
                    _est = _step[1] if isinstance(_step, tuple) else _step
                    if hasattr(_est, 'n_features_in_'):
                        _n_exp = _est.n_features_in_; break
                if _n_exp is None and hasattr(modelo, 'n_features_in_'):
                    _n_exp = modelo.n_features_in_
            except Exception:
                pass

            feats = [f for f in selected_features.get(target, []) if f in dataset.columns]
            if not feats: continue

            x_vals = []
            for f in feats:
                v = ul.get(f)
                if v is None or (isinstance(v, float) and np.isnan(v)):
                    med = treino[f].median() if f in treino.columns else 0.0
                    x_vals.append(0.0 if pd.isna(med) else float(med))
                else:
                    x_vals.append(float(v))

            x = np.array(x_vals)
            if _n_exp is not None:
                if len(x) < _n_exp:
                    x = np.pad(x, (0, _n_exp - len(x)), constant_values=0.0)
                elif len(x) > _n_exp:
                    x = x[:_n_exp]

            x = np.nan_to_num(x.reshape(1, -1), nan=0.0)

            try:
                y_raw  = modelo.predict(x)
                y_pred = float(inv_transform(y_raw, target)[0])
            except Exception as e:
                logger.warning("Predicao falhou | %s | %s | %s", nome_emp, target, e)
                continue

            if np.isnan(y_pred) or np.isinf(y_pred):
                continue

            base = target.replace('TARGET_','').replace('_DFP','')
            rows_pred.append({
                'cnpj':           cnpj,
                'empresa':        nome_emp,
                'setor':          setor,
                'ano_base':       ano_base,
                'target':         target,
                'variavel':       NOME_VAR.get(base, base),
                'algoritmo':      alg_nome,
                'y_pred_2026_bi': round(y_pred / 1e6, 4),
            })

    df_pred26 = (pd.DataFrame(rows_pred).sort_values(['setor','empresa','variavel'])
                 if rows_pred else pd.DataFrame())

if not df_pred26.empty:
    df_pred26.to_csv(PASTA_PAINEL / 'b31_predicao_2026_empresa.csv', index=False)
    print(f"  OK b31_predicao_2026_empresa.csv ({len(df_pred26)} linhas, "
          f"{df_pred26['empresa'].nunique()} empresas)")
    rec = df_pred26[df_pred26['variavel']=='Receita Liquida'][
          ['empresa','setor','y_pred_2026_bi']].sort_values(
          ['setor','y_pred_2026_bi'], ascending=[True,False])
    if not rec.empty:
        print("\n  Receita Liquida predita DFP 2026 (R$ bilhoes):")
        print(rec.to_string(index=False))
else:
    print("  ATENCAO: Nenhuma predicao gerada.")
    print("  Verifique se predicoes_prospectivas.parquet existe em outputs/")
    print("  e se contem registros com targets _DFP.")

print("\n  OK Bloco 2 concluido")


22:18:56 | INFO     | df_prosp: 828 linhas | colunas: ['CNPJ_CIA', 'DT_REFER', 'ORIGEM', 'Target', 'Horizonte', 'Algoritmo', 'y_pred']
22:18:56 | INFO     | Mapeamento de colunas df_prosp: {'cnpj': 'CNPJ_CIA', 'target': 'Target', 'alg': 'Algoritmo', 'pred': 'y_pred'}



  BLOCO 2 - Predicoes DFP 2026 por Empresa
  OK b31_predicao_2026_empresa.csv (207 linhas, 23 empresas)

  OK Bloco 2 concluido


## Bloco 3. KPIs Derivados das Predições 2026

Calcula indicadores financeiros a partir das predições dos targets,
seguindo a abordagem de projeção de demonstrativos de Penman (2013):
*"Financial Statement Analysis and Security Valuation"*, Cap. 15.

**Nota metodológica declarada (conforme boas práticas acadêmicas):**
Estes KPIs são derivados de predições, não de valores observados.
Propagam o erro das predições de forma multiplicativa.
Razões envolvendo duas predições independentes têm variância maior
do que razões com um denominador fixo. Devem ser interpretadas como
estimativas de ordem de grandeza, não como valores precisos.

In [15]:
print("\n" + "="*70)
print("  BLOCO 3 - KPIs Derivados das Predicoes 2026")
print("  NOTA: KPIs derivados propagam o erro das predicoes")
print("="*70)

rows_kpi_der = []

if not df_pred26.empty:
    # Pivota: empresa x variavel -> valor predito
    # aggfunc='last' evita erro se houver duplicatas (ex: mesmo target com anos diferentes)
    pv = (df_pred26.pivot_table(index=['cnpj','empresa','setor','ano_base'],
                                columns='variavel', values='y_pred_2026_bi',
                                aggfunc='last')
                   .reset_index())
    pv.columns.name = None

    REC   = 'Receita Liquida'  if 'Receita Liquida'  in pv.columns else 'Receita L\u00edquida'
    LUC   = 'Lucro Liquido'    if 'Lucro Liquido'    in pv.columns else 'Lucro L\u00edquido'
    EBI   = 'EBITDA'
    AT    = 'Ativo Total'
    AC    = 'Ativo Circulante'
    PC    = 'Passivo Circulante'
    PL    = 'Patrim\u00f4nio L\u00edquido'
    PT    = 'Passivo Total'
    FCO_C = 'FCO'

    # Reconecta com os nomes reais presentes no pivot
    for _nome_real in pv.columns:
        _nl = _nome_real.lower()
        if 'receita' in _nl and 'l' in _nl:  REC = _nome_real
        if 'lucro' in _nl and 'l' in _nl:    LUC = _nome_real
        if 'patrim' in _nl:                  PL  = _nome_real

    for _, row in pv.iterrows():
        def g(col):
            # y_pred_2026_bi esta em bilhoes (valor/1e6). Retorna em R$.
            v = row.get(col)
            if v is None: return None
            try:
                fv = float(v)
            except (TypeError, ValueError):
                return None
            if np.isnan(fv) or np.isinf(fv): return None
            return fv * 1e6   # bilhoes -> R$

        rec = g(REC); luc = g(LUC); ebi = g(EBI)
        at  = g(AT);  ac  = g(AC);  pc  = g(PC)
        pl  = g(PL);  pt  = g(PT);  fco = g(FCO_C)

        def ratio(num, den, scale=1):
            if num is None or den is None: return np.nan
            if abs(den) < 1e-9:           return np.nan
            r = num / den * scale
            return round(r, 4) if np.isfinite(r) else np.nan

        # Endividamento: usa pt/at; fallback pc/at somente se pt ausente
        if pt is not None and at is not None:
            endiv = ratio(pt, at)
        elif pc is not None and at is not None:
            endiv = ratio(pc, at)
        else:
            endiv = np.nan

        rows_kpi_der.append({
            'cnpj':                   row['cnpj'],
            'empresa':                row['empresa'],
            'setor':                  row['setor'],
            'ano_pred':               2026,
            'margem_ebitda_pred':     ratio(ebi, rec),
            'margem_liquida_pred':    ratio(luc, rec),
            'roe_pred':               ratio(luc, pl),
            'roa_pred':               ratio(luc, at),
            'endividamento_pred':     endiv,
            'alavancagem_de_pred':    ratio(pt, pl),
            'liquidez_corrente_pred': ratio(ac, pc),
            'giro_ativo_pred':        ratio(rec, at),
            'fco_receita_pred':       ratio(fco, rec),
            'receita_pred_bi':  round(rec/1e6, 3) if rec is not None else np.nan,
            'lucro_pred_bi':    round(luc/1e6, 3) if luc is not None else np.nan,
            'ebitda_pred_bi':   round(ebi/1e6, 3) if ebi is not None else np.nan,
            'nota_metodologica': 'KPIs derivados de predicoes ML - propagam erro das predicoes individuais',
        })

    df_kpis_der = pd.DataFrame(rows_kpi_der).sort_values(['setor','empresa'])
    df_kpis_der.to_csv(PASTA_PAINEL / 'b31_kpis_derivados_2026.csv', index=False)
    print(f"  OK b31_kpis_derivados_2026.csv ({len(df_kpis_der)} empresas)")

    _m = df_kpis_der[['empresa','setor','margem_ebitda_pred','margem_liquida_pred',
                       'roe_pred','endividamento_pred']].dropna(subset=['margem_ebitda_pred'])
    if not _m.empty:
        print("\n  Margem EBITDA predita 2026 por empresa:")
        print(_m.sort_values(['setor','margem_ebitda_pred'], ascending=[True,False]).to_string(index=False))

    kpi_cols = ['margem_ebitda_pred','margem_liquida_pred','roe_pred','roa_pred',
                'endividamento_pred','liquidez_corrente_pred']
    print("\n  Cobertura de KPIs (empresas com valor valido):")
    for col in kpi_cols:
        n_ok  = df_kpis_der[col].notna().sum()
        n_tot = len(df_kpis_der)
        pct   = n_ok / n_tot * 100 if n_tot else 0
        print(f"    {col:<30} {n_ok}/{n_tot} ({pct:.0f}%)")
else:
    df_kpis_der = pd.DataFrame()
    print("  AVISO: Predicoes 2026 nao disponiveis - Bloco 3 ignorado.")

print("\n  OK Bloco 3 concluido")



  BLOCO 3 - KPIs Derivados das Predicoes 2026
  NOTA: KPIs derivados propagam o erro das predicoes
  OK b31_kpis_derivados_2026.csv (23 empresas)

  Margem EBITDA predita 2026 por empresa:
           empresa       setor  margem_ebitda_pred  margem_liquida_pred  roe_pred  endividamento_pred
            Suzano Commodities              2.3703               0.2006    0.3211              1.0000
     CSN Mineração Commodities              1.7900               0.0000    0.0000              1.0000
              Vale Commodities              1.4978               0.3403    0.5507              1.0000
            Klabin Commodities              1.1281               0.0000    0.0000              1.0000
            Gerdau Commodities              0.7343               0.0255    0.0872              1.0000
         ISA CTEEP     Energia              2.0507               0.2663    0.3211              1.0000
             Taesa     Energia              1.9755               0.2564    0.2438              1

## Bloco 4. Z\'\'-Score Prospectivo (Altman Mercados Emergentes)

Calcula o Z\'\'  sobre os targets preditos para 2026, gerando
classificação de solvência prospectiva por empresa.

Esta é a contribuição metodológica específica do TCC: aplicar
o Z\'\'  sobre *predições* e não apenas sobre dados históricos.


In [16]:
print("\n" + "="*70)
print("  BLOCO 4 - Z''-Score Prospectivo 2026")
print("="*70)

ZONA_SEGURA    = 2.60
ZONA_CINZA_INF = 1.10

def classificar_zona(z):
    if z is None or (isinstance(z, float) and np.isnan(z)): return 'N/D'
    if z > ZONA_SEGURA:      return 'Segura'
    if z >= ZONA_CINZA_INF:  return 'Cinza'
    return 'Insolvencia'

rows_z = []
if not df_kpis_der.empty:
    # Monta lookup rapido: (cnpj, variavel) -> valor em R$
    # y_pred_2026_bi esta em bilhoes -> multiplica por 1e6 para R$
    if not df_pred26.empty:
        _pred_lookup = (df_pred26.copy()
                                 .assign(val_reais=lambda d: d['y_pred_2026_bi'] * 1e6)
                                 .groupby(['cnpj','variavel'])['val_reais']
                                 .last()
                                 .to_dict())
    else:
        _pred_lookup = {}

    for _, row in df_kpis_der.iterrows():
        cnpj = row['cnpj']

        def gv(var):
            v = _pred_lookup.get((cnpj, var))
            if v is None: return None
            fv = float(v)
            return None if (np.isnan(fv) or np.isinf(fv)) else fv

        at_v  = gv('Ativo Total')
        ac_v  = gv('Ativo Circulante')
        pc_v  = gv('Passivo Circulante')
        pl_v  = gv('Patrimonio Liquido') or gv('Patrim\u00f4nio L\u00edquido')
        pt_v  = gv('Passivo Total')
        ebi_v = gv('EBITDA')
        luc_v = gv('Lucro Liquido') or gv('Lucro L\u00edquido')

        # EBIT proxy
        if ebi_v is not None:
            ebit_proxy = ebi_v * 0.85
        elif luc_v is not None:
            ebit_proxy = luc_v
        else:
            ebit_proxy = None

        at_pos = at_v is not None and at_v > 0

        X1 = ((ac_v - pc_v) / at_v
              if at_pos and ac_v is not None and pc_v is not None else None)
        X2 = (pl_v / at_v if at_pos and pl_v is not None else None)
        X3 = (ebit_proxy / at_v if at_pos and ebit_proxy is not None else None)
        X4 = (pl_v / pt_v if pt_v is not None and pt_v > 0 and pl_v is not None else None)

        n_ok = sum(v is not None for v in [X1, X2, X3, X4])

        z_pp = None
        zona = 'N/D'
        if n_ok == 4:
            z_pp = 6.56*X1 + 3.26*X2 + 6.72*X3 + 1.05*X4
            zona = classificar_zona(z_pp)
        elif n_ok >= 3:
            z_pp = (6.56*(X1 or 0) + 3.26*(X2 or 0) +
                    6.72*(X3 or 0) + 1.05*(X4 or 0))
            zona = 'Parcial'
            logger.debug("Z'' parcial (n_ok=%d/4) | %s", n_ok, row['empresa'])

        rows_z.append({
            'empresa':        row['empresa'],
            'setor':          row['setor'],
            'ano_pred':       2026,
            'z_pp_pred':      round(z_pp, 4) if z_pp is not None else np.nan,
            'zona_pred':      zona,
            'X1_CG_AT':       round(X1, 4) if X1 is not None else np.nan,
            'X2_PL_AT':       round(X2, 4) if X2 is not None else np.nan,
            'X3_EBIT_AT':     round(X3, 4) if X3 is not None else np.nan,
            'X4_PL_PT':       round(X4, 4) if X4 is not None else np.nan,
            'componentes_ok': n_ok,
        })

    df_z_pred = pd.DataFrame(rows_z).sort_values(['setor','empresa'])
    df_z_pred.to_csv(PASTA_PAINEL / 'b31_zscore_prospectivo.csv', index=False)
    print(f"  OK b31_zscore_prospectivo.csv ({len(df_z_pred)} empresas)")

    df_z_valido = df_z_pred.dropna(subset=['z_pp_pred'])
    if not df_z_valido.empty:
        print("\n  Z'' prospectivo 2026 por empresa:")
        print(df_z_valido[['empresa','setor','z_pp_pred','zona_pred','componentes_ok']]
              .sort_values(['setor','z_pp_pred'], ascending=[True,False])
              .to_string(index=False))

    dist = df_z_pred['zona_pred'].value_counts()
    print("\n  Distribuicao de zonas (2026 predito):")
    for zona, cnt in dist.items():
        print(f"    {zona:<15} {cnt} ({cnt/len(df_z_pred)*100:.0f}%)")
else:
    df_z_pred = pd.DataFrame()
    print("  AVISO: KPIs derivados nao disponiveis - Bloco 4 ignorado.")

print("\n  OK Bloco 4 concluido")



  BLOCO 4 - Z''-Score Prospectivo 2026
  OK b31_zscore_prospectivo.csv (23 empresas)

  Z'' prospectivo 2026 por empresa:
           empresa       setor  z_pp_pred   zona_pred  componentes_ok
            Suzano Commodities     8.5660      Segura               4
            Klabin Commodities     7.1104      Segura               4
            Gerdau Commodities     7.0567      Segura               4
     CSN Mineração Commodities     6.8896      Segura               4
              Vale Commodities     3.2216      Segura               4
      CPFL Energia     Energia     8.6623      Segura               4
         ISA CTEEP     Energia     8.2732      Segura               4
Equatorial Energia     Energia     8.0294      Segura               4
      Engie Brasil     Energia     7.7040      Segura               4
             Taesa     Energia     6.5226      Segura               4
              Prio    Petróleo     8.6329      Segura               4
          Ultrapar    Petróleo     7.

## Bloco 5. Painel Comparativo Intra-Setor

Para cada setor: histórico 2015–2025 + predição 2026, todas as empresas,
nas três variáveis foco do TCC. Empresa com maior cobertura de dados
é destacada (linha mais espessa e anotada).

Esta figura é a síntese visual do pipeline completo —
dados reais + predição ML — diretamente usável na Seção 6 do TCC.

In [17]:
print("\n" + "="*70)
print("  BLOCO 5 - Painel Comparativo Intra-Setor")
print("="*70)

if df_pred26.empty:
    print("  AVISO: Predicoes 2026 nao disponiveis - Bloco 5 ignorado.")
else:
    ds_dfp = dataset[dataset["ORIGEM"]=="DFP"].copy() if "ORIGEM" in dataset.columns else dataset.copy()

    # Remove espacos extras dos nomes de colunas (" ANO" -> "ANO")
    ds_dfp.columns = ds_dfp.columns.str.strip()

    if "ANO" not in ds_dfp.columns:
        print("  AVISO: Coluna ANO nao encontrada - Bloco 5 ignorado.")
    else:
        ds_dfp = ds_dfp.dropna(subset=["ANO"])
        ds_dfp["ANO"] = ds_dfp["ANO"].astype(int)

        BASES_FOCO = [("DRE_3.01", "Receita L\u00edquida"),
                      ("DRE_3.11", "Lucro L\u00edquido"),
                      ("EBITDA",   "EBITDA")]

        for setor, cnpjs in sorted(setores_empresas.items()):
            empresas_s = [mapa_nome[c] for c in cnpjs if c in mapa_nome]
            if not empresas_s: continue

            fig, axes = plt.subplots(1, len(BASES_FOCO),
                                     figsize=(5.5*len(BASES_FOCO), 5.5), squeeze=False)
            cores_emp = plt.cm.tab10(np.linspace(0, 0.9, max(len(empresas_s), 1)))
            emp_cor   = {e: cores_emp[i] for i, e in enumerate(empresas_s)}

            cob      = {e: ds_dfp[ds_dfp["NOME_CIA"]==e]["ANO"].nunique() for e in empresas_s}
            emp_dest = max(cob, key=cob.get) if cob else None

            for i_b, (base, label_b) in enumerate(BASES_FOCO):
                ax = axes[0][i_b]

                if base not in ds_dfp.columns:
                    ax.set_title(f"{label_b}\n(coluna ausente)", fontsize=9)
                    continue

                for emp in empresas_s:
                    cnpj_e = mapa_cnpj.get(emp)
                    if not cnpj_e: continue

                    hist = (ds_dfp[ds_dfp["NOME_CIA"] == emp][["ANO", base]]
                            .dropna()
                            .drop_duplicates(subset=["ANO"], keep="last")
                            .sort_values("ANO"))
                    if hist.empty: continue

                    destaque = emp == emp_dest
                    lw, ms, alpha, zord = ((2.5, 5, 1.0, 5) if destaque else (1.0, 2, 0.4, 2))

                    ax.plot(hist["ANO"], hist[base] / 1e6,
                            "o-", color=emp_cor[emp], lw=lw, ms=ms,
                            alpha=alpha, zorder=zord,
                            label=emp[:12] if destaque else "_nolegend_")

                    # Conecta com a predicao 2026
                    pred_row = df_pred26[(df_pred26["cnpj"] == cnpj_e) &
                                         (df_pred26["variavel"] == label_b)]
                    if not pred_row.empty:
                        y26        = float(pred_row["y_pred_2026_bi"].iloc[0])
                        ultimo_ano = int(hist["ANO"].iloc[-1])
                        ultimo_val = float(hist[base].iloc[-1]) / 1e6
                        ax.plot([ultimo_ano, 2026], [ultimo_val, y26],
                                "--", color=emp_cor[emp], lw=lw*0.9,
                                alpha=alpha, zorder=zord)
                        ax.scatter([2026], [y26], color=emp_cor[emp],
                                   s=50 if destaque else 20, zorder=zord+1,
                                   marker="*" if destaque else "o")
                        if destaque:
                            ax.annotate(f"{y26:.1f} Bi", xy=(2026, y26),
                                        xytext=(4, 6), textcoords="offset points",
                                        fontsize=7, color=emp_cor[emp], fontweight="bold")

                ax.axvspan(2020, 2021, alpha=0.08, color="gray")
                ax.axhline(0, color="#2c3e50", lw=0.6, ls="--", alpha=0.4)
                ax.set_title(label_b, fontsize=9, fontweight="bold")
                ax.set_xlabel("Ano", fontsize=8)
                ax.set_ylabel("R$ bilhoes", fontsize=8)
                ax.tick_params(labelsize=7)
                ax.grid(alpha=0.2)
                ax.legend(fontsize=7, loc="upper left")

            plt.suptitle(
                f"Setor: {setor} - Historico DFP (2015-2025) + Predicao 2026\n"
                f"Empresa destacada: {emp_dest or ''} (maior cobertura historica)",
                fontsize=10, fontweight="bold"
            )
            plt.tight_layout()
            fname = f"b31_painel_setor_{setor.replace(' ','_')}.png"
            plt.savefig(PASTA_PAINEL / fname, dpi=150, bbox_inches="tight")
            plt.close()
            print(f"  OK {fname}")

print("\n  OK Bloco 5 concluido")



  BLOCO 5 - Painel Comparativo Intra-Setor
  OK b31_painel_setor_Commodities.png
  OK b31_painel_setor_Energia.png
  OK b31_painel_setor_Petróleo.png
  OK b31_painel_setor_Tecnologia.png
  OK b31_painel_setor_Varejo.png

  OK Bloco 5 concluido


## Resumo Final

In [18]:
n_figs = len(list(PASTA_PAINEL.glob("b31_*.png")))
n_csvs = len(list(PASTA_PAINEL.glob("b31_*.csv")))

print("\n" + "═"*70)
print("  RESUMO — Script 3.1")
print("═"*70)
print(f"  Empresas      : {len(mapa_nome)}")
print(f"  Targets DFP   : {len([t for t in TARGETS if t.endswith('_DFP')])}")
print(f"  CSVs gerados  : {n_csvs} em outputs/painel/")
print(f"  Figuras       : {n_figs} em outputs/painel/")
print("─"*70)
print("  Arquivos:")
for f in sorted(PASTA_PAINEL.iterdir()):
    print(f"    • {f.name}")
print("═"*70)
print("  ✅ Script 3.1 concluído — pronto para o Script 4")
logger.info("Script 3.1 concluído | csvs=%d | figs=%d", n_csvs, n_figs)

22:18:59 | INFO     | Script 3.1 concluído | csvs=4 | figs=10



══════════════════════════════════════════════════════════════════════
  RESUMO — Script 3.1
══════════════════════════════════════════════════════════════════════
  Empresas      : 25
  Targets DFP   : 9
  CSVs gerados  : 4 em outputs/painel/
  Figuras       : 10 em outputs/painel/
──────────────────────────────────────────────────────────────────────
  Arquivos:
    • b31_kpis_derivados_2026.csv
    • b31_painel_setor_Commodities.png
    • b31_painel_setor_Energia.png
    • b31_painel_setor_Petróleo.png
    • b31_painel_setor_Tecnologia.png
    • b31_painel_setor_Varejo.png
    • b31_predicao_2026_empresa.csv
    • b31_validacao_holdout.csv
    • b31_validacao_holdout_Commodities.png
    • b31_validacao_holdout_Energia.png
    • b31_validacao_holdout_Petróleo.png
    • b31_validacao_holdout_Tecnologia.png
    • b31_validacao_holdout_Varejo.png
    • b31_zscore_prospectivo.csv
══════════════════════════════════════════════════════════════════════
  ✅ Script 3.1 concluído — pronto par